In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('/content/diabetic_data.csv')

In [3]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1.0,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3.0,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2.0,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2.0,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1.0,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,1


In [5]:
df.drop(columns=['A1Cresult','max_glu_serum'], inplace = True)

In [10]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


In [9]:
df.dropna(inplace=True)

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33242 entries, 0 to 33241
Data columns (total 48 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   encounter_id              33242 non-null  int64  
 1   patient_nbr               33242 non-null  int64  
 2   race                      33242 non-null  object 
 3   gender                    33242 non-null  object 
 4   age                       33242 non-null  object 
 5   weight                    33242 non-null  object 
 6   admission_type_id         33242 non-null  int64  
 7   discharge_disposition_id  33242 non-null  int64  
 8   admission_source_id       33242 non-null  int64  
 9   time_in_hospital          33242 non-null  float64
 10  payer_code                33242 non-null  object 
 11  medical_specialty         33242 non-null  object 
 12  num_lab_procedures        33242 non-null  float64
 13  num_procedures            33242 non-null  float64
 14  num_medicat

In [19]:
df.drop(columns=['encounter_id','patient_nbr','weight','payer_code','medical_specialty'], inplace=True)

In [20]:
df['readmitted'] = (df['readmitted'] == '<30').astype(int)

print(df['readmitted'].value_counts())

readmitted
0    29454
1     3788
Name: count, dtype: int64


In [21]:
age_mapping = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}

df['age'] = df['age'].map(age_mapping)

In [22]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].astype(str).str[:3]

In [23]:
X = df.drop(columns=['readmitted'])
y = df['readmitted']


In [24]:
id_categorical = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

for col in id_categorical:
    if col in X.columns:
        X[col] = X[col].astype(str)

In [25]:
numeric_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X.select_dtypes(
    include=['object']
).columns.tolist()

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [27]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numeric_features
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=True
            ),
            categorical_features
        )
    ]
)

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('preprocessor', preprocessor),
    (
        'classifier',
        LogisticRegression(
            penalty='l2',
            C=1.0,
            max_iter=1000,
            random_state=42
        )
    )
])

In [30]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'time_in_hospital',
                                                   'num_lab_procedures',
                                                   'num_procedures',
                                                   'num_medications',
                                                   'number_outpatient',
                                                   'number_emergency',
                                                   'number_inpatient',
                                                   'number_diagnoses']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['race', 'gender',
                                                   'admission_type_id',
                                                   'discharge_disposi...
                                                   'glimepiride',
                                                   'acetohexamide', 'glipizide',
                                                   'glyburide', 'tolbutamide',
                                                   'pioglitazone',
                                                   'rosiglitazone', 'acarbose',
                                                   'miglitol', 'troglitazone',
                                                   'tolazamide', 'examide',
                                                   'citoglipton', 'insulin',
                                                   'glyburide-metformin',
                                                   'glipizide-metformin',
                                                   'glimepiride-pioglitazone',
                                                   'metformin-rosiglitazone', ...])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [31]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.6475785476615865


In [33]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)


Confusion Matrix:
[[5866   25]
 [ 740   18]]
